# 12.13 GNN expressiveness & the Weisfeiler-Leman test

Message-passing GNNs inherit the strengths and limits of 1-WL refinement. Message-passing GNNs inherit much of the power and limits of 1-WL refinement. This walkthrough implements WL recoloring, a WL isomorphism screen, and aggregation collisions across increasingly noisy graphs. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 1215
random.seed(SEED)
np.random.seed(SEED)


def make_d1_graph():
    graph = nx.Graph()
    graph.add_nodes_from(range(4))
    graph.add_edges_from([(0, 1), (0, 2), (1, 2), (2, 3)])
    labels = {0: 0, 1: 0, 2: 1, 3: 1}
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_karate_rung():
    graph = nx.karate_club_graph()
    labels = {}
    for node, data in graph.nodes(data=True):
        labels[node] = 0 if data.get("club") == "Mr. Hi" else 1
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_sbm_rung(sizes, p_in, p_out, seed, noise_edges=0):
    probs = []
    for row in range(len(sizes)):
        prob_row = []
        for col in range(len(sizes)):
            prob_row.append(p_in if row == col else p_out)
        probs.append(prob_row)
    graph = nx.stochastic_block_model(sizes, probs, seed=seed)
    rng = np.random.default_rng(seed)
    nodes = list(graph.nodes())
    for _ in range(noise_edges):
        u = int(rng.choice(nodes))
        v = int(rng.choice(nodes))
        if u != v:
            graph.add_edge(u, v)
    labels = {}
    offset = 0
    for block, size in enumerate(sizes):
        for node in range(offset, offset + size):
            labels[node] = block
        offset = offset + size
    positions = nx.spring_layout(graph, seed=seed)
    return graph, labels, positions


def make_cora_like_rung():
    graph, labels, positions = make_sbm_rung([18, 16, 14], 0.26, 0.035, SEED + 4, noise_edges=8)
    rng = np.random.default_rng(SEED + 44)
    years = {}
    node_types = {}
    for node in graph.nodes():
        years[node] = int(2016 + rng.integers(0, 7))
        node_types[node] = "paper"
    nx.set_node_attributes(graph, years, "year")
    nx.set_node_attributes(graph, node_types, "kind")
    return graph, labels, positions


def make_large_noisy_rung():
    graph, labels, positions = make_sbm_rung([35, 32, 30, 28], 0.18, 0.045, SEED + 5, noise_edges=55)
    rng = np.random.default_rng(SEED + 55)
    for u, v in graph.edges():
        graph[u][v]["time"] = int(rng.integers(0, 10))
        graph[u][v]["relation"] = "strong" if labels[u] == labels[v] else "weak"
    return graph, labels, positions


def build_graph_ladder():
    d1_graph, d1_labels, d1_pos = make_d1_graph()
    d2_graph, d2_labels, d2_pos = make_karate_rung()
    d3_graph, d3_labels, d3_pos = make_sbm_rung([12, 12, 12], 0.32, 0.04, SEED + 3, noise_edges=5)
    d4_graph, d4_labels, d4_pos = make_cora_like_rung()
    d5_graph, d5_labels, d5_pos = make_large_noisy_rung()
    return [
        {"name": "D1 toy", "graph": d1_graph, "labels": d1_labels, "pos": d1_pos},
        {"name": "D2 karate", "graph": d2_graph, "labels": d2_labels, "pos": d2_pos},
        {"name": "D3 SBM", "graph": d3_graph, "labels": d3_labels, "pos": d3_pos},
        {"name": "D4 Cora-like subset", "graph": d4_graph, "labels": d4_labels, "pos": d4_pos},
        {"name": "D5 larger noisy", "graph": d5_graph, "labels": d5_labels, "pos": d5_pos},
    ]


def partition_from_labels(labels):
    groups = defaultdict(set)
    for node, label in labels.items():
        groups[label].add(node)
    return list(groups.values())


def accuracy_score(y_true, y_pred):
    correct = 0
    total = 0
    for node, truth in y_true.items():
        if node in y_pred:
            correct = correct + int(y_pred[node] == truth)
            total = total + 1
    return correct / max(total, 1)


## The concept, built once (D1): 1-WL recoloring

1-WL updates colors by hashing the current color and neighbor multiset:

$$c_v^{(k+1)}=\operatorname{HASH}(c_v^{(k)},\{\!\{c_u^{(k)}:u\in N(v)\}\!\}).$$

The lesson checks degree signatures, a star's two colors, sum and mean collisions, and persistent 4-cycle symmetry.

In [ ]:

def wl_refine(graph, iterations=3, initial_colors=None):
    if initial_colors is None:
        colors = {node: graph.degree(node) for node in graph.nodes()}
    else:
        colors = dict(initial_colors)
    history = [dict(colors)]
    for _ in range(iterations):
        signatures = {}
        for node in graph.nodes():
            neighbor_colors = sorted(colors[neighbor] for neighbor in graph.neighbors(node))
            signatures[node] = (colors[node], tuple(neighbor_colors))
        palette = {}
        next_colors = {}
        for signature in sorted(set(signatures.values())):
            palette[signature] = len(palette)
        for node, signature in signatures.items():
            next_colors[node] = palette[signature]
        colors = next_colors
        history.append(dict(colors))
    return colors, history


def wl_histogram(graph, iterations=3):
    colors, history = wl_refine(graph, iterations=iterations)
    return Counter(colors.values())


## WL isomorphism screen and aggregation collisions

Matching WL color histograms is a useful necessary screen for many graphs, while non-injective aggregators explain why some message passing layers collapse distinct neighborhoods.

In [ ]:

def wl_isomorphic_candidate(graph_a, graph_b, iterations=3):
    hist_a = wl_histogram(graph_a, iterations=iterations)
    hist_b = wl_histogram(graph_b, iterations=iterations)
    return hist_a == hist_b


def multiset_sum(values):
    return sum(values)


def multiset_mean(values):
    return sum(values) / len(values)


graph = nx.Graph()
graph.add_edges_from([(1, 0), (1, 2), (1, 3)])
first_signatures = [graph.degree(node) for node in sorted(graph.nodes())]
star = nx.star_graph(3)
star_colors, star_history = wl_refine(star, iterations=1)
cycle = nx.cycle_graph(4)
cycle_colors, cycle_history = wl_refine(cycle, iterations=2)
sum_a = multiset_sum([1, 1, 1])
sum_b = multiset_sum([1, 2])
mean_a = multiset_mean([1, 1])
mean_b = multiset_mean([1])
print("degree signatures", first_signatures)
print("star color count", len(set(star_colors.values())))
print("sum collision", sum_a, sum_b)
print("mean collision", mean_a, mean_b)
print("cycle colors", sorted(cycle_colors.values()))
assert first_signatures == [1, 3, 1, 1]
assert len(set(star_colors.values())) == 2
assert sum_a == 3
assert sum_b == 3
assert len([1, 1, 1]) == 3
assert len([1, 2]) == 2
assert mean_a == 1
assert mean_b == 1
assert len(set(cycle_colors.values())) == 1
assert wl_isomorphic_candidate(cycle, nx.cycle_graph(4), iterations=2)


## The dataset ladder

D1 is the four-node toy; D2 karate; D3 noisy SBM; D4 Cora-like subset; D5 larger noisy graph.

In [ ]:

ladder = build_graph_ladder()
for rung in ladder:
    graph = rung["graph"]
    labels = rung["labels"]
    sample_nodes = list(graph.nodes())[:5]
    sample_edges = list(graph.edges())[:5]
    print(rung["name"])
    print("  nodes", graph.number_of_nodes(), "edges", graph.number_of_edges(), "classes", sorted(set(labels.values())))
    print("  sample nodes", sample_nodes)
    print("  sample edges", sample_edges)


## Run the same method across D1-D5

In [ ]:

def wl_predict_labels(graph, labels, iterations=3):
    colors, history = wl_refine(graph, iterations=iterations)
    color_to_label = {}
    for color in set(colors.values()):
        nodes = [node for node, value in colors.items() if value == color]
        votes = [labels[node] for node in nodes]
        color_to_label[color] = Counter(votes).most_common(1)[0][0]
    predictions = {node: color_to_label[color] for node, color in colors.items()}
    return predictions, colors


ladder = build_graph_ladder()
wl_results = []
for rung in ladder:
    predictions, colors = wl_predict_labels(rung["graph"], rung["labels"], iterations=3)
    score = accuracy_score(rung["labels"], predictions)
    rung["predictions"] = predictions
    rung["wl_colors"] = colors
    rung["accuracy"] = score
    wl_results.append(score)
    print(f"{rung['name']}: colors={len(set(colors.values()))} node_accuracy={score:.3f}")


## Results visualization

Top row: graph panels colored by WL colors. Bottom left: node-accuracy curve from WL features.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for index, rung in enumerate(ladder):
    graph = rung["graph"]
    positions = rung["pos"]
    colors = [rung["wl_colors"][node] for node in graph.nodes()]
    nx.draw_networkx(
        graph,
        pos=positions,
        node_color=colors,
        cmap="tab20",
        with_labels=False,
        node_size=55,
        edge_color="#cccccc",
        ax=axes[0, index],
    )
    axes[0, index].set_title(rung["name"])
    axes[0, index].axis("off")
axes[1, 0].plot(range(1, 6), wl_results, marker="o")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("node accuracy")
axes[1, 0].set_title("WL-feature accuracy")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung: mean aggregation is not injective

The neighborhoods $[1,1]$ and $[1]$ have the same mean. Add a count-aware signature to recover the distinction.

In [ ]:

left = [1, 1]
right = [1]
mean_left = multiset_mean(left)
mean_right = multiset_mean(right)
injective_left = (sum(left), len(left))
injective_right = (sum(right), len(right))
print("mean([1,1])", mean_left)
print("mean([1])", mean_right)
print("mean collision", mean_left == mean_right)
print("count-aware signatures", injective_left, injective_right)
print("fixed collision", injective_left == injective_right)



## Evaluate it + practice

- Metric: compare the rung's node accuracy against the no-skill baseline `majority class` on D1-D5.
- Sanity check: rerun with the fixed seed and verify D1 reproduces the asserted lesson numbers before trusting larger rungs.
- Ablation: replace WL/count-aware signatures with pure mean aggregation; the metric should drop or the failure should become visible.
- Failure signal: a high score with a violated split, symmetry, or relation contract is not a real win.
- Inspect the hardest rung by plotting both the graph artifact and the metric curve rather than reading one scalar alone.

Practice prompts:
1. Change only the D3 noise level and predict how the metric curve should move.


2. Replace the D5 seed with another fixed seed and compare the artifact panel.

3. Add one cheap assertion that would catch the lesson pitfall before the metric is printed.